# Fragrantica Perfume Scraper

This notebook contains a Python script using `requests` and `BeautifulSoup` to scrape specific perfume details from a Fragrantica product page. The script is designed to be easily reusable by changing the `target_url` variable.

**Note on Execution:** When run in certain environments (like the sandbox where this was developed), the script may encounter a **403 Forbidden** error due to the website's anti-bot measures. The script includes robust headers to mitigate this, but if the error persists, you may need to run it from a different network or consider using a proxy/VPN.

### Extracted Fields

| Field | Extraction Logic |
| :--- | :--- |
| `name` | Extracted from the main `<h1>` title, after removing the gender part. |
| `gender` | Extracted from the end of the main `<h1>` title (e.g., "for women and men"). |
| `rating` | Extracted from the "Perfume rating X.XX out of 5" text. |
| `rating_count` | Extracted from the "with X votes" text. |
| `main_accords` | Scraped from the "main accords" section, returning a list of strings. |
| `perfumers` | Scraped by looking for links or text containing the perfumer's name. |
| `description` | Extracted from the main descriptive paragraph on the page. |
| `url` | The input URL. |

In [4]:
def scrape_fragrantica(url):
    """
    Scrapes a Fragrantica perfume page for specific details.

    Args:
        url (str): The URL of the Fragrantica perfume page.

    Returns:
        dict: A dictionary containing the extracted perfume data.
    """
    # Using a complete set of headers to mimic a real browser
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.9',
        'Accept-Language': 'en-US,en;q=0.9',
        'Referer': 'https://www.google.com/',
        'Connection': 'keep-alive',
        'Upgrade-Insecure-Requests': '1',
    }
    
    try:
        response = requests.get(url, headers=headers, timeout=15 )
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"Error fetching URL {url}: {e}")
        return {"error": str(e)}

    soup = BeautifulSoup(response.content, 'html.parser')
    data = {"url": url}

    # 1. Name and Gender
    data['name'] = 'N/A'
    data['gender'] = 'N/A'
    try:
        # The main title is usually in an <h1> tag with class 'text-center'
        title_tag = soup.find('h1', class_='text-center')
        if title_tag:
            full_title = title_tag.text.strip()
            
            # Extract gender from the end of the title
            gender_match = re.search(r'for (women and men|men|women)$', full_title, re.IGNORECASE)
            if gender_match:
                data['gender'] = gender_match.group(0).lower()
                # Remove the gender part to get the clean name
                data['name'] = full_title[:gender_match.start()].strip()
            else:
                data['name'] = full_title
                
    except Exception as e:
        print(f"Error extracting name/gender: {e}")

    # 2. Rating and Rating Count (REVISED - using more general text search)
    data['rating'] = 'N/A'
    data['rating_count'] = 'N/A'
    try:
        # Search for the text pattern "Perfume rating X.XX out of 5 with Y votes" anywhere in the text
        rating_text_tag = soup.find(string=re.compile(r'Perfume rating [\d\.]+ out of 5 with [\d,]+ votes'))
        if rating_text_tag:
            rating_text = rating_text_tag.strip()
            
            rating_match = re.search(r'rating ([\d\.]+) out of 5', rating_text)
            count_match = re.search(r'with ([\d,]+) votes', rating_text)
            
            data['rating'] = float(rating_match.group(1)) if rating_match else 'N/A'
            data['rating_count'] = int(count_match.group(1).replace(',', '')) if count_match else 'N/A'
        
        # Fallback: Look for the specific div that contains the rating stars and text
        if data['rating'] == 'N/A':
            rating_div = soup.find('div', class_='rating-stars')
            if rating_div:
                rating_value_tag = rating_div.find('span', itemprop='ratingValue')
                review_count_tag = rating_div.find('span', itemprop='reviewCount')
                
                if rating_value_tag:
                    data['rating'] = float(rating_value_tag.text.strip())
                if review_count_tag:
                    data['rating_count'] = int(review_count_tag.text.strip().replace(',', ''))
                    
    except Exception as e:
        print(f"Error extracting rating/count: {e}")

    # 3. Main Accords
    data['main_accords'] = []
    try:
        # Main accords are usually in a div with class 'accord-box'
        accord_box = soup.find('div', class_='accord-box')
        if accord_box:
            accords = accord_box.find_all('div', class_='accord-bar')
            for accord in accords:
                accord_name_tag = accord.find('span')
                if accord_name_tag:
                    data['main_accords'].append(accord_name_tag.text.strip().lower())
                else:
                    data['main_accords'].append(accord.text.strip().lower())
            
    except Exception as e:
        print(f"Error extracting main accords: {e}")

    # 4. Perfumers
    data['perfumers'] = []
    try:
        # Perfumer information is often a link with a href containing '/perfumer/'
        perfumer_tag = soup.find('a', href=re.compile(r'/perfumer/'))
        if perfumer_tag:
            data['perfumers'].append(perfumer_tag.text.strip())
            
    except Exception as e:
        print(f"Error extracting perfumers: {e}")

    # 5. Description (REVISED - looking for the first <p> tag after the main title block)
    data['description'] = 'N/A'
    try:
        # Find the main title <h1> tag
        title_tag = soup.find('h1', class_='text-center')
        if title_tag:
            # Find the next sibling that is a <p> tag, which is often the main description
            description_p = title_tag.find_next_sibling('p')
            if description_p:
                data['description'] = description_p.text.strip()
            else:
                # Fallback to the original selector if the first one fails
                description_div = soup.find('div', class_='text-content')
                if description_div:
                    data['description'] = description_div.text.strip()
        
    except Exception as e:
        print(f"Error extracting description: {e}")

    return data


In [5]:
# Example Usage
target_url = "https://www.fragrantica.com/perfume/Fragrance-World/Velvet-Rouge-104781.html"

print(f"Scraping data from: {target_url}\n")
perfume_data = scrape_fragrantica(target_url)

# Print the extracted data in a clean JSON format
print(json.dumps(perfume_data, indent=4, ensure_ascii=False))

# Optional: Print in the requested CSV-like format for easy comparison
print("\n--- CSV-like Output ---")
print("name\tgender\trating\trating_count\tmain_accords\tperfumers\tdescription\turl")

# Prepare the description for a single-line output (remove newlines/tabs)
clean_description = perfume_data.get('description', '').replace('\n', ' ').replace('\t', ' ').strip()

print(f"{perfume_data.get('name')}\t"
      f"{perfume_data.get('gender')}\t"
      f"{perfume_data.get('rating')}\t"
      f"{perfume_data.get('rating_count')}\t"
      f"{perfume_data.get('main_accords')}\t"
      f"{perfume_data.get('perfumers')}\t"
      f"{clean_description}\t"
      f"{perfume_data.get('url')}")

Scraping data from: https://www.fragrantica.com/perfume/Fragrance-World/Velvet-Rouge-104781.html

{
    "url": "https://www.fragrantica.com/perfume/Fragrance-World/Velvet-Rouge-104781.html",
    "name": "Velvet Rouge Fragrance World",
    "gender": "for women and men",
    "rating": "N/A",
    "rating_count": "N/A",
    "main_accords": [
        "rose"
    ],
    "perfumers": [],
    "description": "N/A"
}

--- CSV-like Output ---
name	gender	rating	rating_count	main_accords	perfumers	description	url
Velvet Rouge Fragrance World	for women and men	N/A	N/A	['rose']	[]	N/A	https://www.fragrantica.com/perfume/Fragrance-World/Velvet-Rouge-104781.html
